# Knot-selection strategies for K-DAREK

Trains K-DAREK on `10*cos(x)` once per knot-selection strategy
(`Kmean` / `random` / `LHS` / `gw_kmean` / `igw_kmean` / `chebyshev`, all `kan_extend=True`)
and compares them: an error/violation table, an overlay of all fits, and a
per-method grid scattering where each strategy actually placed its knots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from kdarek import DAREK, KDAREK as K2DAREK, Dataset

seed = 12
cos_dataset = Dataset(fx=lambda x: 10 * np.cos(x), n=50, fix=True, seed=seed)
plt.scatter(cos_dataset['train_input'], cos_dataset['train_label'], label='train', alpha=0.5)
plt.scatter(cos_dataset['test_input'], cos_dataset['test_label'], label='test', alpha=0.05)
plt.legend()

## DAREK baseline (single knot-selection method, spline-only)

In [ ]:
x_train, y_train = cos_dataset['train_input'], cos_dataset['train_label']
x, y = cos_dataset['test_input'], cos_dataset['test_label']
xindx = x.sort(dim=0)[1][:, 0]
x, y = x[xindx], y[xindx]

dk = DAREK(width=[1, 5, 1], grid=8, k=3, base_fun='silu', seed=seed, device='cpu',
           symbolic_enabled=False, auto_save=False, extend=True)
dk.fit(cos_dataset, lr=0.1, steps=500, lamb=0.0, nonfixknot=True, seed_knots=42,
       rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)

dk_xg, dk_yg = dk.samples['x'], dk(dk.samples['x']).detach()
dk_hat, dk_u = dk.predict(x, fk=10, f1=10)
dk_lb, dk_ub = (dk_hat - dk_u).detach().flatten().numpy(), (dk_hat + dk_u).detach().flatten().numpy()

plt.plot(x, y, '--', label='True', color='k')
plt.plot(x, dk_hat.detach().numpy().flatten(), label='dk', color='blue')
plt.scatter(dk_xg, dk_yg, label='knots', color='blue')
plt.fill_between(x.flatten(), dk_lb, dk_ub, color='blue', alpha=0.2)
plt.legend()
plt.ylim([-20, 20])

## K-DAREK across 6 knot-selection strategies

In [ ]:
def check_violation(xt, yt, lb, ub, eps=1e-4):
    xt, yt = xt.flatten(), yt.flatten()
    return (1 - np.bitwise_and(lb < (yt + eps), yt < (ub + eps)).sum() / yt.shape[0]).item()

eps = 1e-5
models = {}
for knot_method in ['Kmean', 'random', 'LHS', 'gw_kmean', 'igw_kmean', 'chebyshev']:
    print('knot_method', knot_method)
    kdk = K2DAREK(mlp_width=[1, 5], kan_width=[5, 1], kan_grid=8, kan_k=3, kan_base_fun='silu',
                  kan_seed=seed, device='cpu', L_l=1.0, symbolic_enabled=False, auto_save=False,
                  kan_extend=True)
    kdk.fit(cos_dataset, lr=0.1, steps=1000, lamb=0.0, nonfixknot=True, seed_knots=42,
            rand_method=knot_method, scheduler='dec', step_sch=50, gamma=0.9, verbose=False)

    kdk_xg, kdk_yg = kdk.samples['xi'].numpy().flatten(), kdk(kdk.samples['xi']).detach().numpy().flatten()
    kdk_hat, kdk_u = kdk.predict(x, L_k=10, L_1=10)
    kdk_hat, kdk_u = kdk_hat.detach().numpy().flatten(), kdk_u.detach().numpy().flatten()
    kdk_lb, kdk_ub = (kdk_hat - kdk_u), (kdk_hat + kdk_u)
    xt, yt = x.numpy().flatten(), y.numpy().flatten()

    kdk_er = np.sqrt(((kdk_hat - yt) ** 2).mean())
    kdk_vio = check_violation(xt, yt, kdk_lb, kdk_ub, eps)

    models[knot_method] = {'model': kdk, 'error': kdk_er, 'violation': kdk_vio,
                            'knot_x': kdk_xg, 'knot_y': kdk_yg, 'lb': kdk_lb, 'ub': kdk_ub,
                            'yhat': kdk_hat, 'xt': xt, 'yt': yt}

## Results table + overlay of all fits

In [ ]:
import pandas as pd
import matplotlib.cm as cm

rows = [{'method': k, 'error': m['error'], 'violation': m['violation']} for k, m in models.items()]
df = pd.DataFrame(rows).set_index('method')
print(df.to_string(float_format=lambda v: f'{v:.4f}'))

fig, ax = plt.subplots(figsize=(10, 6))
colors = cm.tab10(np.linspace(0, 1, len(models)))

first_method = next(iter(models))
xt0, yt0 = models[first_method]['xt'], models[first_method]['yt']
sort0 = np.argsort(xt0)
ax.plot(xt0[sort0], yt0[sort0], 'k-', linewidth=2, label='True', zorder=10)

for (knot_method, m), color in zip(models.items(), colors):
    xt_m = m['xt']
    sort_idx = np.argsort(xt_m)
    ax.plot(xt_m[sort_idx], m['yhat'][sort_idx], color=color, linestyle='--', label=f'{knot_method} (est)')
    ax.fill_between(xt_m[sort_idx], m['lb'][sort_idx], m['ub'][sort_idx], color=color, alpha=0.3)

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('True vs estimated with uncertainty bounds, by knot method')
ax.legend(loc='best', fontsize=8)
plt.tight_layout()

## Knot-selection plot: where each strategy places its knots

In [ ]:
n_methods = len(models)
ncols = 3
nrows = int(np.ceil(n_methods / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (knot_method, m) in zip(axes, models.items()):
    xt_m = m['xt']
    sort_idx = np.argsort(xt_m)
    ax.plot(xt_m[sort_idx], m['yt'][sort_idx], 'k-', linewidth=2, label='True')
    ax.plot(xt_m[sort_idx], m['yhat'][sort_idx], 'C1--', linewidth=1.5, label='Estimated')
    ax.fill_between(xt_m[sort_idx], m['lb'][sort_idx], m['ub'][sort_idx], color='C1', alpha=0.2, label='Bound')
    ax.scatter(m['knot_x'], m['knot_y'])
    ax.set_title(f"{knot_method}  (err={m['error']:.3f}, vio={m['violation']:.3f})")
    ax.set_xlabel('x'); ax.set_ylabel('y')

for ax in axes[len(models):]:
    ax.axis('off')

axes[0].legend(loc='best', fontsize=8)
plt.tight_layout()

## Estimated curves only (bounds omitted, for a cleaner comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
linestyles = ['--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]
colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

xt0, yt0 = models[first_method]['xt'], models[first_method]['yt']
sort0 = np.argsort(xt0)
ax.plot(xt0[sort0], yt0[sort0], 'k-', linewidth=2.5, label='True', zorder=10)

for (knot_method, m), color, ls in zip(models.items(), colors, linestyles):
    xt_m = m['xt']
    sort_idx = np.argsort(xt_m)
    ax.plot(xt_m[sort_idx], m['yhat'][sort_idx], color=color, linestyle=ls, linewidth=1.8, label=f'{knot_method}')

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Estimated values by knot method (bounds omitted for clarity)')
ax.legend(loc='best', fontsize=8)
plt.tight_layout()